In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)
import torch 

from auto_circuit.data import load_datasets_from_json
from auto_circuit.experiment_utils import load_tl_model
from auto_circuit.prune_algos.mask_gradient import mask_gradient_prune_scores
from auto_circuit.types import PruneScores
from auto_circuit.utils.graph_utils import patchable_model
from auto_circuit.utils.misc import repo_path_to_abs_path
from auto_circuit.visualize import draw_seq_graph

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import PreTrainedTokenizerFast, AutoTokenizer
import transformer_lens
from transformer_lens import HookedTransformer, HookedTransformerConfig
import json

%load_ext autoreload
%autoreload 2

In [ ]:
TOKENIZER_DIR  = "../model/wordlevel_tokenizer"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, add_bos_token=True)

# --- Load config ---
with open("../model/trained_transformerlens_model/config.json", "r") as f:
    cfg_dict = json.load(f)

# --- Fix dtype string back to actual torch dtype ---
if isinstance(cfg_dict.get("dtype"), str):
    cfg_dict["dtype"] = getattr(torch, cfg_dict["dtype"].replace("torch.", ""))

# --- Rebuild config and model ---
config = HookedTransformerConfig.from_dict(cfg_dict)
model = HookedTransformer(config)
model.load_state_dict(torch.load("../model/trained_transformerlens_model/model_weights.pth"))
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

model.set_tokenizer(tokenizer)

model.cfg.default_prepend_bos, model.cfg.tokenizer_prepends_bos

model.set_use_hook_mlp_in(True)

model.set_use_attn_result(True)
model.set_use_attn_in(True)
model.set_use_split_qkv_input(True)
model.set_use_hook_mlp_in(True)

model.eval()

for param in model.parameters():
    param.requires_grad = False

In [ ]:
# NOTE: we aleady have the json files with the augmented prompts, so we can load them directly in the next step
# do not run, else the 'seq_labels' (which we manually added) key will be overwritten

from data.succession import generate_successor_pairs, create_prompt, create_flipped_prompt, to_json_format, create_augmented_prompts

succession_data, _ = generate_successor_pairs()
task_prompts = create_prompt(succession_data)
flipped_task_prompts = create_flipped_prompt(succession_data)

augmented_data_last = create_augmented_prompts(tokenizer, task_prompts, flipped_task_prompts, truncate=True, which_task='last')
augmented_data_next = create_augmented_prompts(tokenizer, task_prompts, flipped_task_prompts, truncate=True, which_task='next')

to_json_format(augmented_data_last, save_path="../data/succession_augmented_last_equal_len.json")
to_json_format(augmented_data_next, save_path="../data/succession_augmented_next_equal_len.json")

In [ ]:
path = repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/data/succession_augmented_last_equal_len.json")
with open(path, "r") as f:
    data = json.load(f)

clean_prompts = [d["clean"] for d in data["prompts"]]
corrupt_prompts = [d["corrupt"] for d in data["prompts"]]
print("Number of prompts:", len(clean_prompts))

# Quick sanity check that all prompts are of equal length
for i in range(len(clean_prompts)):
    clean_toks = tokenizer(clean_prompts[i], return_tensors="pt", add_special_tokens=True)
    toks = clean_toks["input_ids"][0]
    print(f"Prompt {i}:")
    print("  Clean prompt:", clean_prompts[i])
    print("  Token IDs:", toks.tolist())
    print("  Tokenized length:", len(toks))
    print("---")


In [ ]:
path = repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/data/succession_augmented_last_equal_len.json")
train_loader, test_loader = load_datasets_from_json(
    model=model,
    path=path,
    device=device,
    prepend_bos=False,
    batch_size=1,
    train_test_size=(13, 13),
    return_seq_length=True,
)
auto_model = patchable_model(
    model,
    factorized=True,
    slice_output="last_seq",
    seq_len=test_loader.seq_len,
    separate_qkv=True,
    device=device,
)

attribution_scores: PruneScores = mask_gradient_prune_scores(
    model=auto_model,
    dataloader=train_loader,
    official_edges=None,
    grad_function="logit",
    answer_function="avg_diff",
    mask_val=0.0,
)

fig = draw_seq_graph(auto_model, attribution_scores, 3.5, show_all_seq_pos=True, seq_labels=train_loader.seq_labels, display_ipython=True, orientation='h')
fig.write_image(repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/auto_circuit_exps/test_tokenwise_last.png"))

In [ ]:
# path = repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/data/succession_dataset_uniform.json")
path = repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/data/succession_augmented_next_equal_len.json")
train_loader, test_loader = load_datasets_from_json(
    model=model,
    path=path,
    device=device,
    prepend_bos=False,
    batch_size=1,
    train_test_size=(13, 13),
    return_seq_length=True,
)
auto_model = patchable_model(
    model,
    factorized=True,
    slice_output="last_seq",
    seq_len=test_loader.seq_len,
    separate_qkv=True,
    device=device,
)

attribution_scores: PruneScores = mask_gradient_prune_scores(
    model=auto_model,
    dataloader=train_loader,
    official_edges=None,
    grad_function="logit",
    answer_function="avg_diff",
    mask_val=0.0,
)

fig = draw_seq_graph(auto_model, attribution_scores, 3.5, show_all_seq_pos=True, seq_labels=train_loader.seq_labels, display_ipython=True, orientation='h')
fig.write_image(repo_path_to_abs_path("/home/iustin/Round_and_Round_RoPE/auto_circuit_exps/test_tokenwise_next.png"))